In [1]:
from pathlib import Path
import sys

# Resolve the project from the notebook's working directory (root or any subfolder).
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (start, *start.parents)
     if (folder / "replication_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start Jupyter from the replication project or a subfolder of it. "
        "The project root must contain replication_paths.py."
    )
for source_dir in (PROJECT_ROOT, PROJECT_ROOT / "01_taxonomy", PROJECT_ROOT / "03_judge_prompts",
                   PROJECT_ROOT / "04_llm_evaluation"):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))
from replication_paths import DATASET_DIR, RAW_DEVGPT_DIR, INTERIM_DIR, RESULTS_DIR, HUMAN_FILE, OUTPUT_DIR

import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

r1_raw = pd.read_excel(HUMAN_FILE, sheet_name="Smell_Matrix_R1")
r2_raw = pd.read_excel(HUMAN_FILE, sheet_name="Smell_Matrix_R2")

In [2]:
smell_cols = r1_raw.columns[3:24].tolist()  # The 21 smell columns in the annotation protocol
if len(smell_cols) != 21 or not all(column in r2_raw.columns for column in smell_cols):
    raise ValueError("The two reviewer sheets must contain the same 21 smell columns.")

r1 = r1_raw[r1_raw["Review Status"] == "Reviewed"].set_index("prompt_id")
r2 = r2_raw[r2_raw["Review Status"] == "Reviewed"].set_index("prompt_id")

common_ids = r1.index.intersection(r2.index)

r1 = r1.loc[common_ids, smell_cols].copy()
r2 = r2.loc[common_ids, smell_cols].copy()

print(f"Prompts: {len(common_ids)} | Smells: {len(smell_cols)}")

Prompts: 260 | Smells: 21


In [3]:
LABELS = ["0", "1", "?"]
LABEL_NAMES = ["absent", "present", "cannot_determine"]


def normalize_label(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if not text:
        return pd.NA

    if text == "?":
        return "?"

    try:
        numeric = float(text)
    except ValueError:
        raise ValueError(f"Invalid annotation: {value!r}") from None

    if numeric == 0:
        return "0"
    if numeric == 1:
        return "1"

    raise ValueError(f"Invalid annotation: {value!r}")


if not r1.index.is_unique or not r2.index.is_unique:
    raise ValueError("Duplicate prompt IDs found.")

if not r1.index.equals(r2.index) or not r1.columns.equals(r2.columns):
    raise ValueError("Reviewer matrices are not aligned.")

if r1.empty:
    raise ValueError("No common reviewed prompts found.")

r1 = r1.apply(lambda column: column.map(normalize_label))
r2 = r2.apply(lambda column: column.map(normalize_label))

In [4]:
def prompt_has_smell(row):
    if row.eq("1").any():
        return "1"

    if row.isna().any() or row.eq("?").any():
        return pd.NA

    return "0"


prompt_presence = pd.DataFrame({
    "r1": r1.apply(prompt_has_smell, axis=1),
    "r2": r2.apply(prompt_has_smell, axis=1),
})

valid = prompt_presence.dropna()

agreement = np.nan
kappa = np.nan

if not valid.empty:
    agreement = valid["r1"].eq(valid["r2"]).mean()

    if len(pd.unique(valid.to_numpy().ravel())) > 1:
        kappa = cohen_kappa_score(
            valid["r1"],
            valid["r2"],
            labels=["0", "1"],
        )

print(f"Valid prompts: {len(valid)}")
print(f"Excluded prompts: {len(prompt_presence) - len(valid)}")
print(f"Agreement: {agreement:.2%}")
print(f"Kappa: {kappa:.4f}")

Valid prompts: 260
Excluded prompts: 0
Agreement: 68.08%
Kappa: 0.3494


In [5]:
results = []

for smell in smell_cols:
    paired = pd.DataFrame({
        "r1": r1[smell],
        "r2": r2[smell],
    }).dropna()

    agreement = np.nan
    kappa = np.nan

    if not paired.empty:
        agreement = paired["r1"].eq(paired["r2"]).mean()

        if len(pd.unique(paired.to_numpy().ravel())) > 1:
            kappa = cohen_kappa_score(
                paired["r1"],
                paired["r2"],
                labels=["0", "1", "?"],
            )

    results.append({
        "smell": smell,
        "valid_prompts": len(paired),
        "excluded_prompts": len(r1) - len(paired),
        "agreement_pct": agreement * 100,
        "kappa": kappa,
    })

agreement_by_smell = pd.DataFrame(results).set_index("smell")

display(
    agreement_by_smell.style.format({
        "agreement_pct": "{:.2f}%",
        "kappa": "{:.4f}",
    }, na_rep="—")
)

,valid_prompts,excluded_prompts,agreement_pct,kappa
smell,,,,
Grammar Smell,260,0,76.92%,0.4756
Formatting Smell,260,0,69.23%,0.1294
Readability Smell,260,0,95.00%,0.0511
Ambiguous Instruction,260,0,76.92%,0.2801
Missing Constraints,260,0,88.85%,0.3614
Conflicting Instructions,260,0,95.38%,0.1329
Intent Misalignment,260,0,98.85%,0.0000
Missing Context Smell,260,0,86.92%,0.4492
Irrelevant Context,260,0,94.62%,0.3122


In [6]:
paired = pd.DataFrame({
    "r1": r1.to_numpy().ravel(),
    "r2": r2.to_numpy().ravel(),
}).dropna()

agreement = np.nan
kappa = np.nan

if not paired.empty:
    agreement = paired["r1"].eq(paired["r2"]).mean()

    if len(pd.unique(paired.to_numpy().ravel())) > 1:
        kappa = cohen_kappa_score(
            paired["r1"],
            paired["r2"],
            labels=["0", "1", "?"],
        )

print(f"Classifications: {len(paired)}")
print(f"Agreement: {agreement:.2%}")
print(f"Kappa: {kappa:.4f}")

Classifications: 5460
Agreement: 93.02%
Kappa: 0.3485


In [7]:
smell_counts = pd.DataFrame({
    "R1": r1.eq("1").sum(axis=0),
    "R2": r2.eq("1").sum(axis=0),
})

smell_counts["Combined"] = smell_counts["R1"] + smell_counts["R2"]

smell_counts = (
    smell_counts
    .sort_values("Combined", ascending=False)
    .rename_axis("smell")
)

display(smell_counts)

,R1,R2,Combined
smell,,,
Grammar Smell,105,49,154
Formatting Smell,80,11,91
Ambiguous Instruction,35,33,68
Missing Context Smell,28,27,55
Missing Constraints,14,23,37
Role Separation Smell,31,2,33
Irrelevant Context,10,7,17
Incorrect Information,5,8,13
Readability Smell,4,8,12


In [8]:
present_r1 = r1.eq("1").fillna(False)
present_r2 = r2.eq("1").fillna(False)

presence_comparison = pd.DataFrame({
    "R1_present": present_r1.sum(),
    "R2_present": present_r2.sum(),
    "Both_present": (present_r1 & present_r2).sum(),
    "Only_R1_present": (present_r1 & ~present_r2).sum(),
    "Only_R2_present": (~present_r1 & present_r2).sum(),
})

union = (present_r1 | present_r2).sum()

presence_comparison["Jaccard_pct"] = (
    presence_comparison["Both_present"]
    .div(union.replace(0, np.nan))
    .mul(100)
)

display(
    presence_comparison
    .sort_values("Both_present", ascending=False)
    .style.format({"Jaccard_pct": "{:.2f}%"}, na_rep="—")
)

,R1_present,R2_present,Both_present,Only_R1_present,Only_R2_present,Jaccard_pct
Grammar Smell,105,49,47,58,2,43.93%
Missing Context Smell,28,27,17,11,10,44.74%
Ambiguous Instruction,35,33,14,21,19,25.93%
Formatting Smell,80,11,9,71,2,10.98%
Missing Constraints,14,23,8,6,15,27.59%
Incorrect Information,5,8,3,2,5,30.00%
Irrelevant Context,10,7,3,7,4,21.43%
Prompt Injection,2,2,2,0,0,100.00%
Role Separation Smell,31,2,1,30,1,3.12%
Conflicting Instructions,9,2,1,8,1,10.00%
